# DNTR MMDet3 — Kaggle 训练 Notebook

## 使用前必读

**前置步骤（在 Kaggle 界面完成）：**

1. 右上角 `Session options` → `Accelerator` 选择 **GPU (T4 或 P100)**。
2. 右上角 `Settings` → `Internet` 确保已**开启**（需手机验证的账号）。
3. 将 AI-TOD 数据集以 Kaggle Dataset 形式上传，并在本 Notebook 的 `Add data` 中添加，挂载后路径即 `/kaggle/input/<your-dataset-slug>/`。
4. 修改下方 **[配置区]** 中的 `GITHUB_REPO`、`KAGGLE_DATASET_INPUT_DIR` 两个变量。

**输出目录：** `/kaggle/working/work_dirs/aitod_DNTR_mask_mmdet3_train/`  
训练完成后可在 Notebook 右侧 Output 面板下载 checkpoint。

## [配置区] 修改这里

In [ ]:
# ============================================================
# 必须修改的两个变量
# ============================================================

# 你的 GitHub 仓库地址（SSH 或 HTTPS 均可，推荐 HTTPS）
GITHUB_REPO = 'https://github.com/<your-username>/dntr_mmdet3_migration.git'

# AI-TOD 数据集在 Kaggle 挂载后的路径
# 例如你把 dataset slug 命名为 "ai-tod-dataset"，路径通常是:
#   '/kaggle/input/ai-tod-dataset'
# 或若数据集内有子目录:
#   '/kaggle/input/ai-tod-dataset/AI-TOD'
KAGGLE_DATASET_INPUT_DIR = '/kaggle/input/ai-tod-dataset'

# ============================================================
# 一般不需要修改的变量
# ============================================================

# 克隆后项目在 /kaggle/working/ 下的目录名
REPO_NAME = 'dntr_mmdet3_migration'

# 分支名（默认 main，按需改为 master 或其他）
GIT_BRANCH = 'main'

# 训练输出目录（Kaggle 只有 /kaggle/working/ 可写）
WORK_DIR = '/kaggle/working/work_dirs/aitod_DNTR_mask_mmdet3_train'

print('GitHub repo :', GITHUB_REPO)
print('Dataset dir :', KAGGLE_DATASET_INPUT_DIR)
print('Work dir    :', WORK_DIR)

## Step 1 — 克隆项目代码

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = f'/kaggle/working/{REPO_NAME}'

if Path(PROJECT_DIR).exists():
    print(f'[SKIP] 目录已存在，执行 git pull 更新: {PROJECT_DIR}')
    result = subprocess.run(
        ['git', '-C', PROJECT_DIR, 'pull', 'origin', GIT_BRANCH],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('[WARN] git pull failed:', result.stderr)
else:
    print(f'[CLONE] {GITHUB_REPO} → {PROJECT_DIR}')
    result = subprocess.run(
        ['git', 'clone', '--depth', '1', '-b', GIT_BRANCH, GITHUB_REPO, PROJECT_DIR],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError('git clone 失败:\n' + result.stderr)

print('\n项目文件列表:')
for p in sorted(Path(PROJECT_DIR).iterdir()):
    print(' ', p.name)

## Step 2 — 安装依赖

Kaggle 的 GPU notebook 预装了 PyTorch + CUDA。  
这里只需补装 mmengine、mmcv、mmdet 及其他小依赖。

In [ ]:
import importlib
import subprocess
import sys


def _run(cmd, *, timeout=1200, check=False):
    print('[RUN]', ' '.join(cmd))
    r = subprocess.run(cmd, timeout=timeout, check=check)
    return r.returncode == 0


def _has(mod):
    try:
        importlib.import_module(mod)
        return True
    except Exception:
        return False


def _ver(mod):
    try:
        m = importlib.import_module(mod)
        return getattr(m, '__version__', 'installed')
    except Exception:
        return 'MISSING'


# ---- 基础包 ----
base_pkgs = ['mmengine', 'pycocotools', 'timm', 'torchprofile']
missing_base = [p for p in base_pkgs if not _has(p)]
if missing_base:
    _run([sys.executable, '-m', 'pip', 'install', '-q', '-U', *missing_base], timeout=900)
else:
    print('Base packages already installed.')

# ---- mmcv（用 openmim 安装，自动匹配 torch+cuda 版本）----
if not _has('mmcv'):
    print('Installing openmim + mmcv...')
    _run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'openmim'], timeout=300)
    ok = _run([sys.executable, '-m', 'mim', 'install', 'mmcv'], timeout=1200)
    if not ok:
        raise RuntimeError(
            'mmcv 安装失败。\n'
            '请检查 Kaggle 当前 PyTorch 版本与 mmcv 的兼容性：'
            'https://mmcv.readthedocs.io/en/latest/get_started/installation.html'
        )
else:
    print('mmcv already installed:', _ver('mmcv'))

# ---- mmdet ----
if not _has('mmdet'):
    _run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'mmdet'], timeout=900)
else:
    print('mmdet already installed:', _ver('mmdet'))

# ---- 版本汇总 ----
print('\n===== Package Versions =====')
for m in ['torch', 'torchvision', 'mmengine', 'mmcv', 'mmdet',
          'pycocotools', 'timm', 'torchprofile']:
    print(f'  {m:<15}: {_ver(m)}')

## Step 3 — 环境与 GPU 检查

In [ ]:
import platform
import sys
import torch

print('Python      :', sys.version)
print('Platform    :', platform.platform())
print('PyTorch     :', torch.__version__)
print('CUDA avail  :', torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    total_gb = props.total_memory / 1024 ** 3
    print(f'GPU         : {props.name}')
    print(f'VRAM        : {total_gb:.1f} GB')
    print(f'CUDA version: {torch.version.cuda}')
else:
    print('[WARN] No CUDA GPU detected — training will be very slow on CPU!')

## Step 4 — 配置路径 & 修改 config

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_DIR = f'/kaggle/working/{REPO_NAME}'
DATA_ROOT   = KAGGLE_DATASET_INPUT_DIR
CFG_PATH    = f'{PROJECT_DIR}/configs/aitod_DNTR_mask_mmdet3_train.py'

# 基本健全性检查
assert Path(PROJECT_DIR).exists(), f'项目目录不存在: {PROJECT_DIR}'
assert Path(DATA_ROOT).exists(),   (
    f'数据集目录不存在: {DATA_ROOT}\n'
    '请在 Kaggle Add data 面板添加 AI-TOD dataset，并将上方 KAGGLE_DATASET_INPUT_DIR 改为实际路径。'
)
assert Path(CFG_PATH).exists(),    f'配置文件不存在: {CFG_PATH}'

# 注入 sys.path
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# 注入 PYTHONPATH 供子进程使用
existing_pp = os.environ.get('PYTHONPATH', '')
os.environ['PYTHONPATH'] = (PROJECT_DIR + ':' + existing_pp).rstrip(':')

# 替换 config 中的 data_root / work_dir
cfg_text = Path(CFG_PATH).read_text(encoding='utf-8')

# 用正则替换，兼容 data_root 值的各种写法
import re
cfg_text = re.sub(
    r"data_root\s*=\s*['\"].*?['\"]",
    f"data_root = '{DATA_ROOT}/'",
    cfg_text,
)
cfg_text = re.sub(
    r"work_dir\s*=\s*['\"].*?['\"]",
    f"work_dir = '{WORK_DIR}'",
    cfg_text,
)

Path(CFG_PATH).write_text(cfg_text, encoding='utf-8')
Path(WORK_DIR).mkdir(parents=True, exist_ok=True)

print('Config updated :', CFG_PATH)
print('PROJECT_DIR    :', PROJECT_DIR)
print('DATA_ROOT      :', DATA_ROOT)
print('WORK_DIR       :', WORK_DIR)
print('PYTHONPATH     :', os.environ['PYTHONPATH'])

## Step 5 — 核验数据集文件

In [ ]:
import os
from pathlib import Path

ann_dir = Path(DATA_ROOT) / 'annotations'
train_ann = ann_dir / 'instances_train.json'
val_ann   = ann_dir / 'instances_val.json'

print(f'annotations dir : {ann_dir}  exists={ann_dir.exists()}')
print(f'train annotation: {train_ann}  exists={train_ann.exists()}')
print(f'val   annotation: {val_ann}  exists={val_ann.exists()}')

# 列出图片目录（可选验证）
for split in ['train', 'val', 'test']:
    img_dir = Path(DATA_ROOT) / split
    if img_dir.exists():
        n = len(list(img_dir.iterdir()))
        print(f'{split} images dir: {img_dir}  ({n} items)')

if not train_ann.exists() or not val_ann.exists():
    raise FileNotFoundError(
        'annotations 文件缺失！\n'
        '请确认 Kaggle Dataset 内部目录结构为:\n'
        '  <dataset_root>/\n'
        '    annotations/\n'
        '      instances_train.json\n'
        '      instances_val.json\n'
        '    train/  (图片)\n'
        '    val/    (图片)\n'
        '如子目录不匹配，请修改上方 KAGGLE_DATASET_INPUT_DIR 至正确的子路径。'
    )

print('\n[OK] 数据集结构核验通过。')

## Step 6 — 验证自定义模块可导入

In [ ]:
import importlib

for mod in ['mmengine', 'mmcv', 'mmdet', 'timm', 'torchprofile', 'pycocotools']:
    try:
        m = importlib.import_module(mod)
        print(f'  {mod:<15}: {getattr(m, "__version__", "OK")}')
    except Exception as e:
        raise ImportError(f'{mod} 导入失败: {e}')

try:
    import dntr_custom  # noqa: F401
    print(f'  {"dntr_custom":<15}: OK')
except Exception as e:
    raise ImportError(f'dntr_custom 导入失败: {e}\n项目路径: {PROJECT_DIR}')

print('\n[OK] 所有模块导入正常。')

## Step 7 — 开始训练

默认启用 **AMP（混合精度）** 以节省显存、加速训练。  
训练日志同步写入 `WORK_DIR/launch_train_stdout_stderr.log`，可在训练结束后下载。

In [ ]:
import importlib
import os
import subprocess
from pathlib import Path

# 依赖二次确认（防止 Kaggle session 重启后直接跑到此单元）
for mod in ['mmengine', 'mmcv', 'mmdet']:
    try:
        importlib.import_module(mod)
    except Exception as e:
        raise RuntimeError(f'缺少关键依赖 {mod}: {e}。请重新运行 Step 2。')

cmd = [
    'python', f'{PROJECT_DIR}/tools/train_mmdet3.py',
    CFG_PATH,
    '--work-dir', WORK_DIR,
    '--amp',
]

log_path = Path(WORK_DIR) / 'launch_train_stdout_stderr.log'
log_path.parent.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()

print('Command     :', ' '.join(cmd))
print('PYTHONPATH  :', env.get('PYTHONPATH', '(not set)'))
print('Log file    :', log_path)
print('─' * 60)

with open(log_path, 'w', encoding='utf-8') as log_f:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
        log_f.write(line)
    proc.wait()

print('─' * 60)
print('Return code :', proc.returncode)

if proc.returncode != 0:
    print('\n[ERROR] 训练失败，末尾 80 行日志:')
    tail = log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-80:]
    print('\n'.join(tail))
    raise RuntimeError('Training process exited with non-zero code.')
else:
    print('[OK] 训练成功完成。')
    print('Checkpoint 目录:', WORK_DIR)

## Step 8 — 断点续训（可选）

若 Kaggle session 超时中断，取消下方注释后重新运行本单元即可从最新 checkpoint 恢复训练。

In [ ]:
import os
import subprocess
from pathlib import Path

resume_cmd = [
    'python', f'{PROJECT_DIR}/tools/train_mmdet3.py',
    CFG_PATH,
    '--work-dir', WORK_DIR,
    '--amp',
    '--resume',
]

print('Resume command:', ' '.join(resume_cmd))

# 取消下一行注释以执行续训
# subprocess.run(resume_cmd, check=True, env=os.environ.copy())

## 训练完成后

- Notebook 右侧 **Output** 面板 → `work_dirs/aitod_DNTR_mask_mmdet3_train/` 下载 checkpoint。
- 也可在 `Output` 面板将整个输出目录保存为新的 Kaggle Dataset，方便后续推理或继续训练。